# Statistical masking

Combines spatial k-anonymity, generalisation of the quasi-identifiers AGE and GENDER, suppression, and within-cell spatial perturbation: a point is only retained if its grid cell has population ≥ `K`; AGE is generalised into three bands and GENDER is generalised to `"ALL"` wherever a grid-cell/age-band/gender combination would otherwise fall below k = 5; the exact location is then replaced by a new random point drawn uniformly *within the same grid cell.

**Input:** the original point dataset and population grid.
**Output:** `data/statistical_masking/statmasked_points.shp`.


In [ ]:
#imports
import os
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point


In [ ]:
#Paths
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

ODP_PATH = DATA_DIR / "original" / "2008_eov_illness_stand.shp"
GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop.shp"
OUT_DIR = DATA_DIR / "statistical_masking"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUT_DIR / "statmasked_points.shp"

POP_COL = "VALUE"
ID_COL = "grid_id"

#Parameters
K = 5
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

#Loading ODP and population grid
odp = gpd.read_file(ODP_PATH)
grid = gpd.read_file(GRID_PATH)


In [ ]:
#loading data and reprojecting to a common CRS (EOV / EPSG:23700)

odp = gpd.read_file(ODP_PATH).to_crs(epsg=23700)
grid = gpd.read_file(GRID_PATH).to_crs(odp.crs)

if ID_COL not in grid.columns:
    grid[ID_COL] = grid.index.astype(str)

# Assign original points to grid cells
data = gpd.sjoin(
    odp,
    grid[[ID_COL, POP_COL, "geometry"]],
    how="left",
    predicate="within"
)


#drop records that fall outside the grid (no population)
data = data[data[POP_COL] >= K].copy()


#binning age into groups for k-anonymity
data["AGE_GROUP"] = pd.cut(
    data["AGE"],
    bins=[0, 65, 80, np.inf],
    labels=["<65", "65-79", "80+"],
    right=False
).astype(str)


# 3. Suppress small groups
qi_cols = [ID_COL, "AGE_GROUP", "GENDER"]

#Count the number of records in each quasi-identifier group
data["qi_n"] = (
    data.groupby(qi_cols, dropna=False)[ID_COL]
        .transform("size")
)

# Identify groups that need gender generalisation
needs_gender_generalisation = (
    data.groupby([ID_COL, "AGE_GROUP"])["qi_n"]
        .transform(lambda x: (x < K).any())
)

data.loc[needs_gender_generalisation, "GENDER"] = "ALL"

#Recalculate k after gender generalisation
data["qi_n"] = (
    data.groupby(
        [ID_COL, "AGE_GROUP", "GENDER"],
        dropna=False
    )[ID_COL].transform("size")
)

#Suppress remaining unsafe QI groups
data = data[data["qi_n"] >= K].copy()

#replace original coordinates with random points within the same grid cell
grid_geom = grid.set_index(ID_COL).geometry.to_dict()


def random_point(poly):
    minx, miny, maxx, maxy = poly.bounds

    while True:
        p = Point(
            rng.uniform(minx, maxx),
            rng.uniform(miny, maxy)
        )
        if poly.contains(p):
            return p


data["geometry"] = [
    random_point(grid_geom[cell])
    for cell in data[ID_COL]
]

#drop original attributes that are no longer needed
data = data.drop(
    columns=[
        "AGE",          # exact age
        "lat",          # original latitude
        "long",         # original longitude
        "qi_n",
        "index_right",
        ID_COL,
        POP_COL
    ],
    errors="ignore"
)

#Safety check: original coordinate attributes must not remain
assert "lat" not in data.columns
assert "long" not in data.columns

# VISITS remains unchanged
data.to_file(
    OUTPUT_PATH,
    driver="ESRI Shapefile"
)

#print summary statistics
print("Original records:", len(odp))
print("Final records:", len(data))
print("Suppressed:", len(odp) - len(data))